In [1]:
import json
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

PROCESSED_DIR = Path("/content/drive/MyDrive/tastelet/data/processed")

train_df = pd.read_csv(PROCESSED_DIR / "ratings_train.csv")
test_df = pd.read_csv(PROCESSED_DIR / "ratings_test.csv")
n_movies = len(json.loads((PROCESSED_DIR / "movie2idx.json").read_text()))

print(train_df.shape, test_df.shape, n_movies)

Mounted at /content/drive
(72088, 6) (9012, 6) 3636


In [2]:
import numpy as np
import pandas as pd

def rmse(model_predict, test_df: pd.DataFrame) -> float:
    predicted = model_predict(test_df["user_idx"].to_numpy(), test_df["movie_idx"].to_numpy())
    errors = predicted - test_df["rating"].to_numpy()
    return float(np.sqrt(np.mean(errors**2)))

In [3]:
fake_test = pd.DataFrame({"user_idx": [0, 1], "movie_idx": [0, 1], "rating": [4.0, 2.0]})
always_three = lambda users, movies: np.full(len(users), 3.0)
print(rmse(always_three, fake_test))

1.0


In [4]:
import numpy as np

TOP_K = 10
LIKED_RATING = 4.0

def hit_at_10(score_fn, train_df, test_df) -> float:
    """Share of test users with at least one liked test movie in their top 10."""
    seen_by_user = train_df.groupby("user_idx")["movie_idx"].apply(set)
    liked = test_df[test_df["rating"] >= LIKED_RATING]
    liked_by_user = liked.groupby("user_idx")["movie_idx"].apply(set)

    hits = 0
    for user_idx, liked_movies in liked_by_user.items():
        scores = np.asarray(score_fn(user_idx), dtype=float).copy()
        seen = list(seen_by_user.get(user_idx, set()))
        scores[seen] = -np.inf  # never recommend a movie already rated in train
        top_k = np.argpartition(-scores, TOP_K)[:TOP_K]
        if liked_movies & set(top_k.tolist()):
            hits += 1
    return hits / len(liked_by_user)

In [5]:
def cheating_score_fn(user_idx):
    scores = np.zeros(n_movies)
    liked = test_df[(test_df.user_idx == user_idx) & (test_df.rating >= 4.0)]
    scores[liked.movie_idx] = 1.0
    return scores

print(hit_at_10(cheating_score_fn, train_df, test_df))  # expect 1.0

1.0


In [6]:
print(train_df.shape, test_df.shape)

(72088, 6) (9012, 6)


In [7]:
# Simplest possible baseline: predict the same number (the train average) for every rating.
# Any real model must beat this, or it learned nothing.
global_mean = train_df["rating"].mean()

def predict_global_mean(user_ids: np.ndarray, movie_ids: np.ndarray) -> np.ndarray:
    return np.full(len(user_ids), global_mean)

global_mean_rmse = rmse(predict_global_mean, test_df)

print(f"global mean: {global_mean:.3f}")
print(f"RMSE: {global_mean_rmse:.4f}")

global mean: 3.539
RMSE: 1.0334


In [8]:
# Popularity baseline: rank movies by how many people rated them in train.
rating_counts = np.bincount(train_df["movie_idx"], minlength=n_movies)

def popularity_scores(user_idx: int) -> np.ndarray:
    # Same list for every user — that's the point of this baseline.
    return rating_counts.astype(float)

# For RMSE we need a rating, not a rank: use each movie's mean train rating.
rating_sums = np.bincount(
    train_df["movie_idx"], weights=train_df["rating"], minlength=n_movies
)
global_mean = train_df["rating"].mean()
movie_means = np.where(
    rating_counts > 0, rating_sums / np.maximum(rating_counts, 1), global_mean
)

def popularity_predict(user_ids: np.ndarray, movie_ids: np.ndarray) -> np.ndarray:
    return movie_means[movie_ids]

print("Popularity RMSE:", rmse(popularity_predict, test_df))
print("Popularity Hit@10:", hit_at_10(popularity_scores, train_df, test_df))

Popularity RMSE: 0.9505397724741185
Popularity Hit@10: 0.3855855855855856


In [10]:
from scipy.sparse import csr_matrix

n_users = train_df["user_idx"].max() + 1

user_movie_matrix = csr_matrix(
    (train_df["rating"], (train_df["user_idx"], train_df["movie_idx"])),
    shape=(n_users, n_movies),
)
print(user_movie_matrix.shape, user_movie_matrix.nnz)

(610, 3636) 72088


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(user_movie_matrix.T)
print(item_similarity.shape)
print(item_similarity[0, 0])

(3636, 3636)
0.9999999999999999


In [15]:
def item_knn_scores(user_idx: int) -> np.ndarray:
    user_ratings = user_movie_matrix[user_idx].toarray().ravel()
    return user_ratings @ item_similarity

print(item_knn_scores(0).shape)
print(hit_at_10(item_knn_scores, train_df, test_df))

(3636,)
0.5441441441441441


In [16]:
TOP_K_NEIGHBOURS = 20
fallback_rating = train_df["rating"].mean()

def item_knn_predict(user_ids: np.ndarray, movie_ids: np.ndarray) -> np.ndarray:
    predictions = np.empty(len(user_ids))
    for i, (user_idx, movie_idx) in enumerate(zip(user_ids, movie_ids)):
        user_row = user_movie_matrix[user_idx]
        similarities = item_similarity[movie_idx, user_row.indices]
        nearest = np.argsort(similarities)[-TOP_K_NEIGHBOURS:]
        top_similarities = similarities[nearest]
        weight_total = top_similarities.sum()
        if weight_total > 0:
            predictions[i] = top_similarities @ user_row.data[nearest] / weight_total
        else:
            predictions[i] = fallback_rating
    return predictions

print(rmse(item_knn_predict, test_df))

0.8582074599106518


In [18]:
import pandas as pd

results = pd.DataFrame(
    [
        {"model": "global_mean", "rmse": 1.0334, "hit_at_10": None},
        {"model": "popularity", "rmse": None, "hit_at_10": 0.3856},
        {"model": "item_knn", "rmse": 0.8582, "hit_at_10": 0.5441},
    ]
)
results.to_csv("/content/drive/MyDrive/tastelet/results/results.csv", index=False)
results

,model,rmse,hit_at_10
0,global_mean,1.0334,NaN
1,popularity,NaN,0.3856
2,item_knn,0.8582,0.5441
